# Retrieval Augmented Generation (RAG) and Vector Databases

In [1]:
!pip install getenv openai==1.12.0

In [26]:
import os
import pandas as pd
import numpy as np
import openai
import uuid

## Creating our Knowledge base

Creating a Azure Cosmos DB database


In [2]:
pip install azure-cosmos

Note: you may need to restart the kernel to use updated packages.


In [4]:
## create your cosmoss db on Azure CLI using the following commands
## az login
## az group create -n <resource-group-name> -l <location>
## az cosmosdb create -n <cosmos-db-name> -r <resource-group-name>
## az cosmosdb list-keys -n <cosmos-db-name> -g <resource-group-name>

## Once done navigate to data explorer and create a new database and a new container


In [29]:
from azure.cosmos import CosmosClient
from dotenv import load_dotenv

load_dotenv("/etc/.env")  # Load environment variables from .env file
# Initialize Cosmos Client
url = os.getenv('COSMOS_DB_ENDPOINT')
key = os.getenv('COSMOS_DB_KEY')
#print(key)
client = CosmosClient(url, credential=key)

# Select database
database_name = 'rag-cosmos-db'
database = client.get_database_client(database_name)

# Select container
container_name = 'data'
container = database.get_container_client(container_name)



In [10]:
! pwd

/home/kaix/courses/generative-ai-for-beginners/15-rag-and-vector-databases


In [13]:
import pandas as pd

# Initialize an empty DataFrame
df = pd.DataFrame(columns=['path', 'text'])


# splitting our data into chunks
# data_paths= ["data/frameworks.md?WT.mc_id=academic-105485-koreyst", "data/own_framework.md?WT.mc_id=academic-105485-koreyst", "data/perceptron.md?WT.mc_id=academic-105485-koreyst"]
data_paths= ["data/frameworks.md", "data/own_framework.md", "data/perceptron.md"]

rows = []
for path in data_paths:
    with open(path, 'r', encoding='utf-8') as file:
        file_content = file.read()
    # 收集每一行数据
    rows.append({'path': path, 'text': file_content})

# 一次性合并所有数据，避免多次concat
if rows:
    df = pd.concat([df, pd.DataFrame(rows)], ignore_index=True)

df.head()

,path,text
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...
1,data/own_framework.md,# Introduction to Neural Networks. Multi-Layer...
2,data/perceptron.md,# Introduction to Neural Networks: Perceptron\...


In [14]:
def split_text(text, max_length, min_length):
    words = text.split()
    chunks = []
    current_chunk = []

    for word in words:
        current_chunk.append(word)
        if len(' '.join(current_chunk)) < max_length and len(' '.join(current_chunk)) > min_length:
            chunks.append(' '.join(current_chunk))
            current_chunk = []

    # If the last chunk didn't reach the minimum length, add it anyway
    if current_chunk:
        chunks.append(' '.join(current_chunk))

    return chunks

# Assuming analyzed_df is a pandas DataFrame and 'output_content' is a column in that DataFrame
splitted_df = df.copy()
splitted_df['chunks'] = splitted_df['text'].apply(lambda x: split_text(x, 400, 300))

splitted_df

,path,text,chunks
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,[# Neural Network Frameworks As we have learne...
1,data/own_framework.md,# Introduction to Neural Networks. Multi-Layer...,[# Introduction to Neural Networks. Multi-Laye...
2,data/perceptron.md,# Introduction to Neural Networks: Perceptron\...,[# Introduction to Neural Networks: Perceptron...


In [18]:
# Assuming 'chunks' is a column of lists in the DataFrame splitted_df, we will split the chunks into different rows
flattened_df = splitted_df.explode('chunks')

flattened_df.head(30)

,path,text,chunks
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,# Neural Network Frameworks As we have learned...
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,descent optimization While the `numpy` library...
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,should give us the opportunity to compute grad...
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,those computations on GPUs is very important. ...
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,"API, there is also higher-level API, called Ke..."
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,**computational graphs**. This graph defines h...
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,optimizing model parameters. **High-level APIs...
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,you to construct typical neural networks very ...
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,understand that you can use both APIs together...
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,use your own low-level training loop to perfor...


## Converting our text to embeddings

Converting out text  to embeddings, and storing them in our database in chunks

In [19]:
openai.api_type = "azure"
openai.api_key = os.getenv("AZURE_OPENAI_API_KEY") 
openai.api_base = os.getenv("AZURE_OPENAI_ENDPOINT") 
openai.api_version = "2025-03-01-preview"



In [20]:
from openai import OpenAI
client = OpenAI(api_key=os.getenv("AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT"))

In [21]:
def create_embeddings(text, model="text-embedding-3-large"):
    # Create embeddings for each document chunk
    embeddings = openai.embeddings.create(input = text, model=model).data[0].embedding
    return embeddings

#embeddings for the first chunk
create_embeddings(flattened_df['chunks'][0])

[-0.003241153433918953,
 -0.014502998441457748,
 -0.016438385471701622,
 -0.027715738862752914,
 0.03828592970967293,
 0.0038614696823060513,
 0.006767652463167906,
 0.029824813827872276,
 -0.03530840948224068,
 0.0025432973634451628,
 0.002115278970450163,
 0.0036288511473685503,
 -0.0033528103958815336,
 0.0017585970927029848,
 -0.02203363925218582,
 -0.003213239135220647,
 -0.021351290866732597,
 -0.024688594043254852,
 0.002395972143858671,
 -0.020780600607395172,
 0.02736836113035679,
 -0.014105995185673237,
 -0.009168276563286781,
 -0.044166531413793564,
 -0.008609992451965809,
 -0.008423897437751293,
 0.025854788720607758,
 0.0018919651629403234,
 -0.035482101142406464,
 0.03369558975100517,
 0.037268612533807755,
 -0.010160783305764198,
 -0.012133389711380005,
 -0.023683682084083557,
 -0.01132077444344759,
 0.0007839248864911497,
 0.04027094319462776,
 -0.02100391499698162,
 -0.03198351338505745,
 0.013684180565178394,
 0.06972356885671616,
 0.03310008347034454,
 0.021127978339

In [22]:
cat = create_embeddings("cat")
cat

[-0.047014132142066956,
 0.01779354363679886,
 0.00048539438284933567,
 -0.00619917269796133,
 0.0363706536591053,
 0.028861453756690025,
 -0.03388935327529907,
 -0.003599519608542323,
 0.009574232622981071,
 0.025923069566488266,
 -0.034999411553144455,
 -0.02678826078772545,
 0.001146785682067275,
 -0.040517039597034454,
 -0.008129526861011982,
 0.0016742662992328405,
 -0.009655853733420372,
 -0.006137956399470568,
 0.012888075783848763,
 0.0108801806345582,
 -0.006072658579796553,
 -0.03953757882118225,
 0.020225871354341507,
 0.007158228196203709,
 0.02678826078772545,
 0.006244064308702946,
 0.010790396481752396,
 0.025204798206686974,
 -0.005150333046913147,
 -0.004203520715236664,
 0.0357176810503006,
 0.06830108910799026,
 0.015834620222449303,
 0.028877777978777885,
 -0.019507599994540215,
 -0.02440490573644638,
 0.011051585897803307,
 -0.005574766080826521,
 0.010276179760694504,
 0.0009202852961607277,
 -0.019785113632678986,
 -0.006231821142137051,
 -0.050115760415792465,
 

In [23]:
# create embeddings for the whole data chunks and store them in a list

embeddings = []
for chunk in flattened_df['chunks']:
    embeddings.append(create_embeddings(chunk))

# store the embeddings in the dataframe
flattened_df['embeddings'] = embeddings

flattened_df.head()

,path,text,chunks,embeddings
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,# Neural Network Frameworks As we have learned...,"[-0.003241153433918953, -0.014502998441457748,..."
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,descent optimization While the `numpy` library...,"[-0.0009641986107453704, -0.02039271593093872,..."
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,should give us the opportunity to compute grad...,"[-0.026377493515610695, -0.015362123027443886,..."
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,those computations on GPUs is very important. ...,"[-0.02388629876077175, 0.0005730547709390521, ..."
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,"API, there is also higher-level API, called Ke...","[-0.009057976305484772, 0.004360453225672245, ..."


# 可以把embeding存放到数据库里，比如cosmos

In [27]:
container = database.get_container_client(container_name)
# Upload the DataFrame to Cosmos DB
for index, row in flattened_df.iterrows():
    item = {
        # id  需要是唯一的自增
        # Cosmos DB requires a unique identifier for each item
        # Here we use the index of the DataFrame as the id
        # You can also use a UUID or any other unique identifier
        'id': str(uuid.uuid4()),
        'path': row['path'],
        'text': row['text'],
        'chunks': row['chunks'],
        'embeddings': row['embeddings']
    }
    container.upsert_item(item)

## upload dataset into cosmos db
![img](./images/image.png)
在 Azure Cosmos DB 中查找最相似向量，通常有两种方式：

1. **使用 Azure Cosmos DB for MongoDB vCore（支持向量检索）**
2. **手动在代码中实现向量相似度计算（如欧氏距离或余弦相似度）**

下面分别给出代码示例和性能说明。

---

### 方式一：使用 Azure Cosmos DB for MongoDB vCore（推荐，原生支持向量检索）

如果你使用的是 Cosmos DB for MongoDB vCore，可以直接用 `$vectorSearch` 查询：

```python
from pymongo import MongoClient

client = MongoClient("<your-cosmos-mongodb-connection-string>")
db = client["your-db-name"]
collection = db["your-collection-name"]

query_vector = [0.1, 0.2, 0.3, ...]  # 你的查询向量

results = collection.aggregate([
    {
        "$vectorSearch": {
            "queryVector": query_vector,
            "path": "embeddings",  # 嵌入向量字段名
            "numCandidates": 100,
            "limit": 5
        }
    }
])

for doc in results:
    print(doc)
```

**性能说明：**
- 这种方式利用数据库原生索引，检索速度快，适合大规模数据。
- 支持高维向量，查询延迟低，适合生产环境。

---

### 方式二：在 Python 代码中手动计算相似度

如果你的 Cosmos DB 不支持原生向量检索，可以先查出所有数据，然后在本地用 numpy 计算最相似的向量：

```python
import numpy as np
from azure.cosmos import CosmosClient

# 连接 Cosmos DB
client = CosmosClient(url, credential=key)
database = client.get_database_client(database_name)
container = database.get_container_client(container_name)

# 获取所有嵌入
items = list(container.read_all_items())
embeddings = np.array([item['embeddings'] for item in items])
query_vector = np.array([0.1, 0.2, 0.3, ...])  # 你的查询向量

# 计算欧氏距离
distances = np.linalg.norm(embeddings - query_vector, axis=1)
top_k_indices = distances.argsort()[:5]

# 输出最相似的5个向量
for idx in top_k_indices:
    print(items[idx])
```

**性能说明：**
- 适合数据量较小（几千~几万条），全部拉取到本地后用 numpy 计算，速度较快。
- 数据量大时建议分页拉取或用原生向量检索服务。

---

**总结：**
- 推荐使用 Cosmos DB for MongoDB vCore 的原生向量检索，性能最佳。
- 若无原生支持，可本地拉取数据后用 numpy 计算相似度，适合小规模数据实验和原型开发。




# Retrieval

Vector search and similiarity between our prompt and the database

### Creating an search index and reranking

In [17]:
from sklearn.neighbors import NearestNeighbors

embeddings = flattened_df['embeddings'].to_list()

# Create the search index
nbrs = NearestNeighbors(n_neighbors=5, algorithm='ball_tree').fit(embeddings)

# To query the index, you can use the kneighbors method
distances, indices = nbrs.kneighbors(embeddings)

# Store the indices and distances in the DataFrame
flattened_df['indices'] = indices.tolist()
flattened_df['distances'] = distances.tolist()

flattened_df.head()

,path,text,chunks,embeddings,indices,distances
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,# Neural Network Frameworks As we have learned...,"[-0.016977494582533836, 0.0028917337767779827,...","[0, 2, 11, 3, 1]","[0.0, 0.5220072028343841, 0.5281003720111753, ..."
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,descent optimization While the `numpy` library...,"[-0.014787919819355011, 0.0016925617819651961,...","[1, 0, 32, 2, 50]","[0.0, 0.5689486562368801, 0.5917805129945245, ..."
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,should give us the opportunity to compute grad...,"[-0.03673850744962692, -0.02062208764255047, 0...","[2, 3, 0, 5, 1]","[0.0, 0.5052294707599493, 0.5220072028343841, ..."
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,those computations on GPUs is very important. ...,"[-0.03166744112968445, -0.011117876507341862, ...","[3, 2, 0, 10, 11]","[0.0, 0.5052294707599493, 0.5456879720601056, ..."
0,data/frameworks.md,# Neural Network Frameworks\n\nAs we have lear...,"API, there is also higher-level API, called Ke...","[-0.007904806174337864, -0.03335562348365784, ...","[4, 12, 10, 9, 8]","[0.0, 0.5192304344185765, 0.5523440479637329, ..."


In [18]:
# Your text question
question = "what is a perceptron?"

# Convert the question to a query vector
query_vector = create_embeddings(question)  # You need to define this function

# Find the most similar documents
distances, indices = nbrs.kneighbors([query_vector])

index = []
# Print the most similar documents
for i in range(3):
    index = indices[0][i]
    for index in indices[0]:
        print(flattened_df['chunks'].iloc[index])
        print(flattened_df['path'].iloc[index])
        print(flattened_df['distances'].iloc[index])
    else:
        print(f"Index {index} not found in DataFrame")

in our model, in which case the input vector would be a vector of size N. A perceptron is a **binary classification** model, i.e. it can distinguish between two classes of input data. We will assume that for each input vector x the output of our perceptron would be either +1 or -1, depending on the class.
data/perceptron.md
[0.0, 0.5349479188905069, 0.5355415711920977, 0.5439405604626569, 0.5535213920359319]
# Introduction to Neural Networks: Perceptron One of the first attempts to implement something similar to a modern neural network was done by Frank Rosenblatt from Cornell Aeronautical Laboratory in 1957. It was a hardware implementation called "Mark-1", designed to recognize primitive geometric figures,
data/perceptron.md
[0.0, 0.4573465617700431, 0.5237117623258072, 0.5634745620918584, 0.5671484849463262]
user to adjust the resistance of a circuit. > The New York Times wrote about perceptron at that time: *the embryo of an electronic computer that [the Navy] expects will be able 

## Putting it all together to answer a question

In [22]:
import os
import openai

openai.api_type = "azure"
openai.api_base = os.getenv("AZURE_OPENAI_ENDPOINT")
openai.api_version = "2023-07-01-preview"
openai.api_key = os.getenv("AZURE_OPENAI_API_KEY")

In [24]:
user_input = "what is a perceptron?"

def chatbot(user_input):
    # Convert the question to a query vector
    query_vector = create_embeddings(user_input)

    # Find the most similar documents
    distances, indices = nbrs.kneighbors([query_vector])

    # add documents to query  to provide context
    history = []
    for index in indices[0]:
        history.append(flattened_df['chunks'].iloc[index])

    # combine the history and the user input
    history.append(user_input)

    # create a message object
    messages=[
        {"role": "system", "content": "You are an AI assiatant that helps with AI questions."},
        {"role": "user", "content": history[-1]}
    ]

    # use chat completion to generate a response
    response = openai.chat.completions.create(
        model="gpt-35-turbo-1106",
        temperature=0.7,
        max_tokens=800,
        messages=messages
    )

    return response.choices[0].message

chatbot(user_input)

ChatCompletionMessage(content='A perceptron is a type of artificial neural network model, which is a fundamental unit of a neural network. It is a simple algorithm used for binary classification tasks. The perceptron takes multiple input values, applies weights to these inputs, and produces a single output value. The output is determined by applying a step function to the weighted sum of the inputs. Perceptrons are often used as building blocks for more complex neural network architectures.', role='assistant', function_call=None, tool_calls=None)

## Testing and evaluation

A basic example of how you can use Mean Average Precision (MAP) to evaluate the responses of your model based on their relevance.

In [25]:
from sklearn.metrics import average_precision_score

# Define your test cases
test_cases = [
    {
        "query": "What is a perceptron?",
        "relevant_responses": ["A perceptron is a type of artificial neuron.", "It's a binary classifier used in machine learning."],
        "irrelevant_responses": ["A perceptron is a type of fruit.", "It's a type of car."]
    },
    {
        "query": "What is machine learning?",
        "relevant_responses": ["Machine learning is a method of data analysis that automates analytical model building.", "It's a branch of artificial intelligence based on the idea that systems can learn from data, identify patterns and make decisions with minimal human intervention."],
        "irrelevant_responses": ["Machine learning is a type of fruit.", "It's a type of car."]
    },
    {
        "query": "What is deep learning?",
        "relevant_responses": ["Deep learning is a subset of machine learning in artificial intelligence (AI) that has networks capable of learning unsupervised from data that is unstructured or unlabeled.", "It's a type of machine learning."],
        "irrelevant_responses": ["Deep learning is a type of fruit.", "It's a type of car."]
    },
    {
        "query": "What is a neural network?",
        "relevant_responses": ["A neural network is a series of algorithms that endeavors to recognize underlying relationships in a set of data through a process that mimics the way the human brain operates.", "It's a type of machine learning."],
        "irrelevant_responses": ["A neural network is a type of fruit.", "It's a type of car."]
    }
]

# Initialize the total average precision
total_average_precision = 0

# Test the RAG application
for test_case in test_cases:
    query = test_case["query"]
    relevant_responses = test_case["relevant_responses"]
    irrelevant_responses = test_case["irrelevant_responses"]

    # Generate a response using your RAG application
    response = chatbot(query) 

    # Create a list of all responses and a list of true binary labels
    all_responses = relevant_responses + irrelevant_responses
    true_labels = [1] * len(relevant_responses) + [0] * len(irrelevant_responses)

    # Create a list of predicted scores based on whether the response is the generated response
    predicted_scores = [1 if resp == response else 0 for resp in all_responses]

    # Calculate the average precision for this query
    average_precision = average_precision_score(true_labels, predicted_scores)

    # Add the average precision to the total average precision
    total_average_precision += average_precision

# Calculate the mean average precision
mean_average_precision = total_average_precision / len(test_cases)

In [26]:
mean_average_precision

0.5